In [ ]:
import requests
import pandas as pd
import json
import time
from getpass import getpass
from pathlib import Path

In [ ]:
API_KEY = getpass("Enter API key: ")
BASE_URL = "https://www.steamwebapi.com/steam/api/apps"
headers = {
    "X-Api-Key": API_KEY
}

In [ ]:
# Test
params = {
    "appid": "730"
}
response = requests.get(
    BASE_URL,
    headers=headers,
    params=params,
    timeout=30
)

print("HTTP status", response.status_code)
data = response.json()
print("Response keys:", data.keys())
print(json.dumps(data, indent=2)[:10000])
print(type(data))

In [ ]:
def get_all_games(api_key, limit=500, delay=0.5):
    """
    Download the whole Steam game catalog
    --
    Parameters:
    api_key = string (SteamWebAPI key)
    limit = int (number of games per request)
    delay = float (seconds to wait between requests)
    --
    Returns:
    list containing game dictionaries
    """

    all_games = []
    cursor = None
    page_number = 0

    while True:
        params = {
            "limit": limit,
            "sort": "appid",
            "order": "asc"
        }

        # Add cursor after first request
        if cursor is not None:
            params["cursor"] = cursor

        response = requests.get(
            BASE_URL,
            headers=headers,
            params=params,
            timeout=60
        )

        page_number += 1

        print(
            f"Page {page_number}: "
            f"HTTP {response.status_code}"
        )

        # Check HTTP status
        if response.status_code != 200:
            raise RuntimeError(
                f"API request failed with HTTP "
                f"{response.status_code}: {response.text}"
            )

        data = response.json()
        if data.get("error"):
            raise RuntimeError(
                f"SteamWebAPI returned an error: "
                f"{data['error']}"
            )

        # Extract games
        games = data.get("data", [])
        print(f" Games returned: {len(games)}")
        all_games.extend(games)

        # Get next cursor
        pagination = data.get("pagination", {})
        next_cursor = pagination.get("nextcursor")

        # Stop when no next page exists
        if not next_cursor:
            print("Reached final page.")
            break

        cursor = next_cursor

        # Small pause to not overwhelm API
        time.sleep(delay)

    print(f"\nTotal games downloaded: {len(all_games):,}")
    return all_games

games = get_all_games(API_KEY)

In [ ]:
# ORIGINAL DOWNLOADED DATA
Path("data/raw").mkdir(parents=True, exist_ok=True)

raw_file = Path("data/raw/steam_games_raw.json")

with open(raw_file, "w", encoding="utf-8") as f:
    json.dump(games, f, ensure_ascii=False, indent=2)

print(f"Saved raw data to: {raw_file}")

In [ ]:
# CONVERT TO PANDAS
df = pd.json_normalize(games)
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()
df.info()
df.columns.tolist()

In [ ]:
# SAVE AS CSV
Path("data/processed").mkdir(parents=True, exist_ok=True)

df.to_csv(
    "data/processed/steam_games.csv",
    index=False
)

print("CSV saved.")